# Client Statistics Dashboard

Fetches and visualizes client billing and LLM usage statistics from the Andromeda database.

## Sections
1. **Setup** — packages, DB connection, config
2. **Current Month Summary** — records & revenue per client
3. **Month-over-Month Growth** — records and revenue growth %
4. **Revenue Trend** — line chart per client
5. **Cumulative YTD Revenue** — running total
6. **Daily Volume Heatmap** — document volume by day
7. **Harvest Job Efficiency** — records per job
8. **Pareto / Top Clients** — 80/20 volume analysis
9. **LLM Cost per Client** — token usage & cost in EUR
10. **Model Usage Over Time** — which models, volume and cost
11. **Monthly LLM Cost Trend** — cumulative cost line

## 1. Setup

In [ ]:
# Uncomment on first run
# !pip install psycopg2-binary pandas matplotlib seaborn

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import date

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

In [ ]:
# ── SECRETS ──────────────────────────────────────────────────
# Loaded from the .env file in THIS project folder (git-ignored).
# First run:  cp .env.example .env  and fill it in.
# Start Jupyter from this folder so the kernel's working directory matches.
import os
from pathlib import Path

from dotenv import load_dotenv

ENV_PATH = Path.cwd() / '.env'
if not ENV_PATH.is_file():
    raise RuntimeError(
        f'No .env found at {ENV_PATH}. Copy .env.example to .env in this '
        f'project folder and fill it in (and start Jupyter from this folder).'
    )
load_dotenv(ENV_PATH)


def env(name, default=None, required=True):
    """Read a setting from this folder's .env / the environment."""
    val = os.getenv(name, default)
    if required and not val:
        raise RuntimeError(
            f'Missing required setting: {name}. Add it to {ENV_PATH} '
            f'(see .env.example).'
        )
    return val

# ── DB CONFIG ─────────────────────────────────────────────────
DB = dict(
    host     = env('ANDROMEDA_DB_HOST'),
    port     = int(env('ANDROMEDA_DB_PORT', 5432)),
    dbname   = env('ANDROMEDA_DB_NAME'),
    user     = env('ANDROMEDA_DB_USER'),
    password = env('ANDROMEDA_DB_PASSWORD'),
    sslmode  = env('ANDROMEDA_DB_SSLMODE', 'require'),
)

# ── REPORT PERIOD ─────────────────────────────────────────────
# REPORT_YEAR  = date.today().year
# REPORT_MONTH = date.today().month


REPORT_YEAR  = 2026
REPORT_MONTH = 7

# ── STATE FILTER (billing) ─────────────────────────────────────
# 12=DUPLICATE_DETECTED, 18=DATA_UPLOADED, 20=ATTACHMENT_UPLOADED
STATE_FILTER = 'r.state_id IN (12, 18, 19, 20)'       # current
# STATE_FILTER = 'r.state_id >= 11'               # legacy (pre-2025)

# ── ERROR STATES ───────────────────────────────────────────────
# is_final=true AND is_ok=false
ERROR_STATES = (2, 3, 5, 9, 10, 17, 21)

# ── UNBILLED STATES (harvested but not billed) ─────────────────
# state >= 11 AND NOT IN billing states
UNBILLED_STATES = (11, 13, 14, 15, 16, 17, 21)

# ── LLM PRICING (EUR per 1M tokens) ───────────────────────────
PRICE_INPUT        = 2.101   # €/1M input tokens
PRICE_CACHED_INPUT = 0.526   # €/1M cached input tokens
PRICE_OUTPUT       = 8.41    # €/1M output tokens
PRICE_TRAINING     = 26.3    # €/1M training tokens
PRICE_HOSTING_HOUR = 1.5     # €/hour hosting

print(f'Report period: {REPORT_YEAR}-{REPORT_MONTH:02d}')
print(f'Billing filter: {STATE_FILTER}')
print(f'LLM pricing:    input={PRICE_INPUT} | cached={PRICE_CACHED_INPUT} | output={PRICE_OUTPUT} €/1M tokens')

In [ ]:
def query(sql, params=None):
    with psycopg2.connect(**DB) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            cols = [desc.name for desc in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)

## 2. Current Month Summary

In [ ]:
SQL_MONTHLY_SUMMARY = f"""
WITH per_client AS (
    SELECT
        c.name AS client_name,
        COUNT(DISTINCT r.job_harvest_id) AS jobs_harvested,
        COUNT(r.record_id) AS records_created,
        GREATEST(2990,
            CASE
                WHEN COUNT(r.record_id) <= 500  THEN COUNT(r.record_id) * 10
                WHEN COUNT(r.record_id) <= 1000 THEN 5000  + (COUNT(r.record_id) - 500)  * 9
                WHEN COUNT(r.record_id) <= 1500 THEN 9500  + (COUNT(r.record_id) - 1000) * 8
                WHEN COUNT(r.record_id) <= 2000 THEN 13500 + (COUNT(r.record_id) - 1500) * 7
                WHEN COUNT(r.record_id) <= 3000 THEN 17000 + (COUNT(r.record_id) - 2000) * 6
                WHEN COUNT(r.record_id) <= 5000 THEN 23000 + (COUNT(r.record_id) - 3000) * 5
                ELSE NULL
            END
        ) AS monthly_price_czk
    FROM record r
    INNER JOIN client c ON r.client_id = c.client_id
    WHERE r.is_deleted = false
      AND {STATE_FILTER}
      AND EXTRACT(YEAR  FROM r.created_at) = %(year)s
      AND EXTRACT(MONTH FROM r.created_at) = %(month)s
    GROUP BY c.name
)
SELECT * FROM per_client
UNION ALL
SELECT 'ZZ TOTAL', SUM(jobs_harvested), SUM(records_created), SUM(monthly_price_czk)
FROM per_client
ORDER BY client_name;
"""

df_summary = query(SQL_MONTHLY_SUMMARY, {'year': REPORT_YEAR, 'month': REPORT_MONTH})
df_summary

In [ ]:
clients = df_summary[df_summary.client_name != 'ZZ TOTAL'].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(clients.client_name, clients.records_created)
axes[0].set_title(f'Records Created — {REPORT_YEAR}-{REPORT_MONTH:02d}')
axes[0].set_xlabel('Records')

axes[1].barh(clients.client_name, clients.monthly_price_czk)
axes[1].set_title(f'Monthly Price (CZK) — {REPORT_YEAR}-{REPORT_MONTH:02d}')
axes[1].set_xlabel('CZK')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

## 3. Month-over-Month Growth

In [ ]:
SQL_MOM = f"""
WITH monthly_per_client AS (
    SELECT
        c.name                              AS client_name,
        DATE_TRUNC('month', r.created_at)   AS month,
        COUNT(DISTINCT r.job_harvest_id)    AS jobs_harvested,
        COUNT(r.record_id)                  AS records_created,
        GREATEST(2990,
            CASE
                WHEN COUNT(r.record_id) <= 500  THEN COUNT(r.record_id) * 10
                WHEN COUNT(r.record_id) <= 1000 THEN 5000  + (COUNT(r.record_id) - 500)  * 9
                WHEN COUNT(r.record_id) <= 1500 THEN 9500  + (COUNT(r.record_id) - 1000) * 8
                WHEN COUNT(r.record_id) <= 2000 THEN 13500 + (COUNT(r.record_id) - 1500) * 7
                WHEN COUNT(r.record_id) <= 3000 THEN 17000 + (COUNT(r.record_id) - 2000) * 6
                WHEN COUNT(r.record_id) <= 5000 THEN 23000 + (COUNT(r.record_id) - 3000) * 5
                ELSE NULL
            END
        ) AS monthly_price_czk
    FROM record r
    INNER JOIN client c ON r.client_id = c.client_id
    WHERE r.is_deleted = false
      AND {STATE_FILTER}
    GROUP BY c.name, DATE_TRUNC('month', r.created_at)
)
SELECT
    client_name,
    month,
    records_created,
    monthly_price_czk,
    LAG(records_created)   OVER (PARTITION BY client_name ORDER BY month) AS prev_month_records,
    LAG(monthly_price_czk) OVER (PARTITION BY client_name ORDER BY month) AS prev_month_price_czk,
    ROUND((records_created - LAG(records_created) OVER (PARTITION BY client_name ORDER BY month))
        * 100.0 / NULLIF(LAG(records_created) OVER (PARTITION BY client_name ORDER BY month), 0), 1)
        AS records_growth_pct,
    ROUND((monthly_price_czk - LAG(monthly_price_czk) OVER (PARTITION BY client_name ORDER BY month))
        * 100.0 / NULLIF(LAG(monthly_price_czk) OVER (PARTITION BY client_name ORDER BY month), 0), 1)
        AS revenue_growth_pct
FROM monthly_per_client
ORDER BY client_name, month;
"""

df_mom = query(SQL_MOM)
df_mom['month'] = pd.to_datetime(df_mom['month'], utc=True).dt.normalize().dt.tz_localize(None) + pd.offsets.MonthBegin(0)
df_mom['records_growth_pct'] = pd.to_numeric(df_mom['records_growth_pct'], errors='coerce')
df_mom['revenue_growth_pct'] = pd.to_numeric(df_mom['revenue_growth_pct'], errors='coerce')
df_mom.tail(20)

In [ ]:
# Heatmap: revenue growth % by client × month
pivot = df_mom.pivot(index='client_name', columns='month', values='revenue_growth_pct')
pivot.columns = pivot.columns.strftime('%Y-%m')

fig, ax = plt.subplots(figsize=(14, max(4, len(pivot) * 0.6)))
sns.heatmap(
    pivot, annot=True, fmt='.0f', center=0, cmap='RdYlGn',
    linewidths=0.5, ax=ax, cbar_kws={'label': 'Revenue growth %'}
)
ax.set_title('Month-over-Month Revenue Growth % per Client')
ax.set_ylabel('')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 4. Revenue Trend (line chart per client)

In [ ]:
SQL_TREND = f"""
SELECT
    c.name                              AS client_name,
    DATE_TRUNC('month', r.created_at)   AS month,
    COUNT(r.record_id)                  AS records_created,
    GREATEST(2990,
        CASE
            WHEN COUNT(r.record_id) <= 500  THEN COUNT(r.record_id) * 10
            WHEN COUNT(r.record_id) <= 1000 THEN 5000  + (COUNT(r.record_id) - 500)  * 9
            WHEN COUNT(r.record_id) <= 1500 THEN 9500  + (COUNT(r.record_id) - 1000) * 8
            WHEN COUNT(r.record_id) <= 2000 THEN 13500 + (COUNT(r.record_id) - 1500) * 7
            WHEN COUNT(r.record_id) <= 3000 THEN 17000 + (COUNT(r.record_id) - 2000) * 6
            WHEN COUNT(r.record_id) <= 5000 THEN 23000 + (COUNT(r.record_id) - 3000) * 5
            ELSE NULL
        END
    ) AS monthly_price_czk
FROM record r
INNER JOIN client c ON r.client_id = c.client_id
WHERE r.is_deleted = false
  AND {STATE_FILTER}
GROUP BY c.name, DATE_TRUNC('month', r.created_at)
ORDER BY month, client_name;
"""

df_trend = query(SQL_TREND)
df_trend['month'] = pd.to_datetime(df_trend['month'], utc=True).dt.normalize().dt.tz_localize(None) + pd.offsets.MonthBegin(0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for client, grp in df_trend.groupby('client_name'):
    axes[0].plot(grp['month'], grp['records_created'], marker='o', label=client)
    axes[1].plot(grp['month'], grp['monthly_price_czk'], marker='o', label=client)
axes[0].set_title('Records Created per Month per Client'); axes[0].set_ylabel('Records'); axes[0].legend(fontsize=8)
axes[1].set_title('Revenue (CZK) per Month per Client'); axes[1].set_ylabel('CZK')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')); axes[1].legend(fontsize=8)
for ax in axes: ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 5. Cumulative YTD Revenue

In [ ]:
SQL_CUMULATIVE = f"""
WITH per_client_month AS (
    SELECT
        c.name                              AS client_name,
        DATE_TRUNC('month', r.created_at)   AS month,
        GREATEST(2990,
            CASE
                WHEN COUNT(r.record_id) <= 500  THEN COUNT(r.record_id) * 10
                WHEN COUNT(r.record_id) <= 1000 THEN 5000  + (COUNT(r.record_id) - 500)  * 9
                WHEN COUNT(r.record_id) <= 1500 THEN 9500  + (COUNT(r.record_id) - 1000) * 8
                WHEN COUNT(r.record_id) <= 2000 THEN 13500 + (COUNT(r.record_id) - 1500) * 7
                WHEN COUNT(r.record_id) <= 3000 THEN 17000 + (COUNT(r.record_id) - 2000) * 6
                WHEN COUNT(r.record_id) <= 5000 THEN 23000 + (COUNT(r.record_id) - 3000) * 5
                ELSE NULL
            END
        ) AS monthly_price_czk
    FROM record r
    INNER JOIN client c ON r.client_id = c.client_id
    WHERE r.is_deleted = false
      AND {STATE_FILTER}
      AND EXTRACT(YEAR FROM r.created_at) = %(year)s
    GROUP BY c.name, DATE_TRUNC('month', r.created_at)
),
monthly_totals AS (
    SELECT month, SUM(monthly_price_czk) AS monthly_revenue_czk
    FROM per_client_month
    GROUP BY month
)
SELECT
    month,
    monthly_revenue_czk,
    SUM(monthly_revenue_czk) OVER (ORDER BY month) AS cumulative_revenue_czk
FROM monthly_totals
ORDER BY month;
"""

df_cum = query(SQL_CUMULATIVE, {'year': REPORT_YEAR})
df_cum['month'] = pd.to_datetime(df_cum['month'], utc=True).dt.normalize().dt.tz_localize(None) + pd.offsets.MonthBegin(0)
df_cum['monthly_revenue_czk']    = pd.to_numeric(df_cum['monthly_revenue_czk'],    errors='coerce')
df_cum['cumulative_revenue_czk'] = pd.to_numeric(df_cum['cumulative_revenue_czk'], errors='coerce')

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_cum['month'], df_cum['monthly_revenue_czk'], width=20, alpha=0.6, label='Monthly revenue')
ax2 = ax.twinx()
ax2.plot(df_cum['month'], df_cum['cumulative_revenue_czk'], color='tab:red', marker='o', label='Cumulative YTD')
ax.set_title(f'Monthly & Cumulative YTD Revenue {REPORT_YEAR} (CZK)')
ax.set_ylabel('Monthly (CZK)'); ax2.set_ylabel('Cumulative (CZK)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.tick_params(axis='x', rotation=30)
l1, lb1 = ax.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1+l2, lb1+lb2, loc='upper left')
plt.tight_layout(); plt.show()

## 6. Daily Volume Heatmap

In [ ]:
SQL_DAILY = f"""
SELECT
    c.name             AS client_name,
    DATE(r.created_at) AS day,
    COUNT(r.record_id) AS records_created
FROM record r
INNER JOIN client c ON r.client_id = c.client_id
WHERE r.is_deleted = false
  AND {STATE_FILTER}
  AND EXTRACT(YEAR  FROM r.created_at) = %(year)s
  AND EXTRACT(MONTH FROM r.created_at) = %(month)s
GROUP BY c.name, DATE(r.created_at)
ORDER BY day, client_name;
"""

df_daily = query(SQL_DAILY, {'year': REPORT_YEAR, 'month': REPORT_MONTH})
df_daily['day'] = pd.to_datetime(df_daily['day']).dt.normalize()

pivot_daily = df_daily.pivot_table(index='client_name', columns='day', values='records_created', fill_value=0)
pivot_daily.columns = pivot_daily.columns.strftime('%d')

fig, ax = plt.subplots(figsize=(16, max(4, len(pivot_daily) * 0.6)))
sns.heatmap(pivot_daily, cmap='YlOrRd', linewidths=0.3, ax=ax, cbar_kws={'label': 'Records'})
ax.set_title(f'Daily Document Volume per Client — {REPORT_YEAR}-{REPORT_MONTH:02d}')
ax.set_xlabel('Day'); ax.set_ylabel('')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## 7. Harvest Job Efficiency

In [ ]:
SQL_EFFICIENCY = f"""
SELECT
    c.name                                          AS client_name,
    COUNT(DISTINCT r.job_harvest_id)                AS jobs_harvested,
    COUNT(r.record_id)                              AS records_created,
    ROUND(COUNT(r.record_id)::numeric /
          NULLIF(COUNT(DISTINCT r.job_harvest_id), 0), 1) AS records_per_job
FROM record r
INNER JOIN client c ON r.client_id = c.client_id
WHERE r.is_deleted = false
  AND {STATE_FILTER}
GROUP BY c.name
ORDER BY records_per_job DESC;
"""

df_eff = query(SQL_EFFICIENCY)
df_eff['records_per_job'] = pd.to_numeric(df_eff['records_per_job'], errors='coerce')

fig, ax = plt.subplots(figsize=(10, max(4, len(df_eff) * 0.5)))
bars = ax.barh(df_eff['client_name'], df_eff['records_per_job'])
ax.bar_label(bars, fmt='%.1f', padding=3)
ax.set_title('Records per Harvest Job (all time)'); ax.set_xlabel('Records / Job')
plt.tight_layout(); plt.show()
df_eff

## 8. Pareto — Top Clients by Volume

In [ ]:
SQL_PARETO = f"""
WITH per_client AS (
    SELECT
        c.name              AS client_name,
        COUNT(r.record_id)  AS records_created
    FROM record r
    INNER JOIN client c ON r.client_id = c.client_id
    WHERE r.is_deleted = false
      AND {STATE_FILTER}
    GROUP BY c.name
),
ranked AS (
    SELECT
        client_name,
        records_created,
        RANK() OVER (ORDER BY records_created DESC) AS rank,
        ROUND(records_created * 100.0 / SUM(records_created) OVER (), 1) AS pct_of_total,
        ROUND(SUM(records_created) OVER (ORDER BY records_created DESC)
            * 100.0 / SUM(records_created) OVER (), 1) AS cumulative_pct
    FROM per_client
)
SELECT rank, client_name, records_created, pct_of_total, cumulative_pct
FROM ranked
ORDER BY rank;
"""

df_pareto = query(SQL_PARETO)
df_pareto['pct_of_total']   = pd.to_numeric(df_pareto['pct_of_total'],   errors='coerce')
df_pareto['cumulative_pct'] = pd.to_numeric(df_pareto['cumulative_pct'], errors='coerce')

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_pareto['client_name'], df_pareto['records_created'], alpha=0.7)
ax2 = ax.twinx()
ax2.plot(df_pareto['client_name'], df_pareto['cumulative_pct'], color='tab:red', marker='o')
ax2.axhline(80, color='tab:red', linestyle='--', alpha=0.4, label='80 %')
ax.set_title('Pareto: Records per Client (all time)'); ax.set_ylabel('Records')
ax2.set_ylabel('Cumulative %'); ax2.set_ylim(0, 105)
ax.tick_params(axis='x', rotation=30); ax2.legend()
plt.tight_layout(); plt.show()
df_pareto

## 9. LLM Cost per Client per Month

In [ ]:
SQL_LLM_COST = f"""
WITH cost_per_call AS (
    SELECT
        lu.record_id,
        lu.invoked_at,
        lu.prompt_tokens,
        lu.completion_tokens,
        lu.total_tokens,
        lu.elapsed_time_seconds,
        lu.model_name,
        (lu.prompt_tokens     * {PRICE_INPUT}  / 1000000.0 +
         lu.completion_tokens * {PRICE_OUTPUT} / 1000000.0) AS cost_eur
    FROM llm_data.llm_usage lu
)
SELECT
    c.name                              AS client_name,
    DATE_TRUNC('month', cpc.invoked_at) AS month,
    COUNT(DISTINCT r.record_id)         AS records_processed,
    COUNT(cpc.record_id)                AS llm_calls,
    SUM(cpc.prompt_tokens)              AS total_prompt_tokens,
    SUM(cpc.completion_tokens)          AS total_completion_tokens,
    SUM(cpc.total_tokens)               AS total_tokens,
    ROUND(SUM(cpc.cost_eur)::numeric, 4) AS total_cost_eur
FROM cost_per_call cpc
INNER JOIN record r ON cpc.record_id = r.record_id
INNER JOIN client c ON r.client_id   = c.client_id
GROUP BY c.name, DATE_TRUNC('month', cpc.invoked_at)
ORDER BY month DESC, total_cost_eur DESC;
"""

df_llm = query(SQL_LLM_COST)
df_llm['month'] = pd.to_datetime(df_llm['month'], utc=True).dt.normalize().dt.tz_localize(None) + pd.offsets.MonthBegin(0)
df_llm['total_cost_eur'] = pd.to_numeric(df_llm['total_cost_eur'], errors='coerce')

pivot_llm = df_llm.pivot_table(index='client_name', columns='month', values='total_cost_eur', fill_value=0)
pivot_llm.columns = pivot_llm.columns.strftime('%Y-%m')

fig, ax = plt.subplots(figsize=(14, max(4, len(pivot_llm) * 0.6)))
sns.heatmap(pivot_llm, annot=True, fmt='.2f', cmap='Blues', ax=ax, cbar_kws={'label': 'Cost (EUR)'})
ax.set_title('LLM Cost (EUR) per Client per Month')
ax.set_ylabel('')
plt.tight_layout()
plt.show()
df_llm

## 10. Model Usage Over Time

In [ ]:
SQL_MODELS = f"""
SELECT
    DATE_TRUNC('month', lu.invoked_at)              AS month,
    lu.model_name,
    COUNT(lu.usage_id)                              AS llm_calls,
    SUM(lu.total_tokens)                            AS total_tokens,
    ROUND(AVG(lu.elapsed_time_seconds)::numeric, 2) AS avg_elapsed_sec,
    ROUND((SUM(lu.prompt_tokens)     * {PRICE_INPUT}  / 1000000.0 +
           SUM(lu.completion_tokens) * {PRICE_OUTPUT} / 1000000.0)::numeric, 4) AS total_cost_eur
FROM llm_data.llm_usage lu
GROUP BY DATE_TRUNC('month', lu.invoked_at), lu.model_name
ORDER BY month DESC, llm_calls DESC;
"""

df_models = query(SQL_MODELS)
df_models['month'] = pd.to_datetime(df_models['month'], utc=True).dt.normalize().dt.tz_localize(None) + pd.offsets.MonthBegin(0)
for col in ('avg_elapsed_sec', 'total_cost_eur'):
    df_models[col] = pd.to_numeric(df_models[col], errors='coerce')

pivot_calls = df_models.pivot_table(index='month', columns='model_name', values='llm_calls', fill_value=0)
pivot_calls.index = pivot_calls.index.strftime('%Y-%m')   # categorical labels -> avoids ts-plot freq error

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
pivot_calls.plot(kind='bar', stacked=True, ax=axes[0])
axes[0].set_title('LLM Calls per Model per Month'); axes[0].set_ylabel('Calls')
axes[0].tick_params(axis='x', rotation=30); axes[0].legend(fontsize=8)

pivot_cost = df_models.pivot_table(index='month', columns='model_name', values='total_cost_eur', fill_value=0)
pivot_cost.index = pivot_cost.index.strftime('%Y-%m')
pivot_cost.plot(kind='bar', stacked=True, ax=axes[1])
axes[1].set_title('LLM Cost (EUR) per Model per Month'); axes[1].set_ylabel('EUR')
axes[1].tick_params(axis='x', rotation=30); axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()
df_models

## 11. Monthly LLM Cost Trend (all clients)

In [ ]:
SQL_LLM_TREND = f"""
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', lu.invoked_at)  AS month,
        SUM(lu.prompt_tokens)               AS prompt_tokens,
        SUM(lu.completion_tokens)           AS completion_tokens,
        SUM(lu.total_tokens)                AS total_tokens,
        COUNT(lu.usage_id)                  AS llm_calls,
        ROUND((SUM(lu.prompt_tokens)     * {PRICE_INPUT}  / 1000000.0 +
               SUM(lu.completion_tokens) * {PRICE_OUTPUT} / 1000000.0)::numeric, 4) AS cost_eur
    FROM llm_data.llm_usage lu
    GROUP BY DATE_TRUNC('month', lu.invoked_at)
)
SELECT
    month,
    prompt_tokens,
    completion_tokens,
    total_tokens,
    llm_calls,
    cost_eur,
    ROUND(SUM(cost_eur) OVER (ORDER BY month)::numeric, 4) AS cumulative_cost_eur
FROM monthly
ORDER BY month;
"""

df_llm_trend = query(SQL_LLM_TREND)
df_llm_trend['month'] = pd.to_datetime(df_llm_trend['month'], utc=True).dt.normalize().dt.tz_localize(None) + pd.offsets.MonthBegin(0)
df_llm_trend['cost_eur'] = pd.to_numeric(df_llm_trend['cost_eur'], errors='coerce')
df_llm_trend['cumulative_cost_eur'] = pd.to_numeric(df_llm_trend['cumulative_cost_eur'], errors='coerce')

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_llm_trend['month'], df_llm_trend['cost_eur'], width=20, alpha=0.6, label='Monthly cost')
ax2 = ax.twinx()
ax2.plot(df_llm_trend['month'], df_llm_trend['cumulative_cost_eur'], color='tab:red', marker='o', label='Cumulative')
ax.set_title('Monthly & Cumulative LLM Cost (EUR)')
ax.set_ylabel('Monthly cost (EUR)'); ax2.set_ylabel('Cumulative (EUR)')
ax.tick_params(axis='x', rotation=30)
l1, lb1 = ax.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1+l2, lb1+lb2, loc='upper left')
plt.tight_layout(); plt.show()
df_llm_trend

## 12. Pipeline Funnel

In [ ]:
SQL_FUNNEL = """
SELECT
    c.name AS client_name,
    COUNT(*)                                                                        AS total,
    SUM(CASE WHEN r.state_id NOT IN (2,3)             THEN 1 ELSE 0 END)           AS survived_interception,
    SUM(CASE WHEN r.state_id NOT IN (2,3,5)           THEN 1 ELSE 0 END)           AS read_ok,
    SUM(CASE WHEN r.state_id NOT IN (2,3,5,9,10)      THEN 1 ELSE 0 END)           AS prompted_ok,
    SUM(CASE WHEN r.state_id >= 11                     THEN 1 ELSE 0 END)           AS harvested,
    SUM(CASE WHEN r.state_id IN (12,18,20)             THEN 1 ELSE 0 END)           AS billed
FROM record r
INNER JOIN client c ON r.client_id = c.client_id
WHERE r.is_deleted = false
  AND EXTRACT(YEAR  FROM r.created_at) = %(year)s
  AND EXTRACT(MONTH FROM r.created_at) = %(month)s
GROUP BY c.name
ORDER BY billed DESC;
"""

df_funnel = query(SQL_FUNNEL, {'year': REPORT_YEAR, 'month': REPORT_MONTH})
stages = ['total', 'survived_interception', 'read_ok', 'prompted_ok', 'harvested', 'billed']
stage_labels = ['Intercepted', 'Survived\nInterception', 'Read OK', 'Prompted OK', 'Harvested', 'Billed']

fig, ax = plt.subplots(figsize=(14, max(5, len(df_funnel) * 0.5)))
x = range(len(stages))
for _, row in df_funnel.iterrows():
    ax.plot(x, [row[s] for s in stages], marker='o', label=row['client_name'])
ax.set_xticks(range(len(stages)))
ax.set_xticklabels(stage_labels)
ax.set_title(f'Pipeline Funnel per Client — {REPORT_YEAR}-{REPORT_MONTH:02d}')
ax.set_ylabel('Records')
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout(); plt.show()
df_funnel

## 13. Error Analysis

In [ ]:
SQL_UNBILLED = """
SELECT
    c.name AS client_name,
    DATE_TRUNC('month', r.created_at) AS month,
    COUNT(*) AS unbilled_records,
    SUM(CASE WHEN r.state_id = 11 THEN 1 ELSE 0 END) AS harvested,
    SUM(CASE WHEN r.state_id = 13 THEN 1 ELSE 0 END) AS informed,
    SUM(CASE WHEN r.state_id = 14 THEN 1 ELSE 0 END) AS requested,
    SUM(CASE WHEN r.state_id = 15 THEN 1 ELSE 0 END) AS downloaded,
    SUM(CASE WHEN r.state_id = 16 THEN 1 ELSE 0 END) AS accounting_applied,
    SUM(CASE WHEN r.state_id = 17 THEN 1 ELSE 0 END) AS accounting_failed,
    SUM(CASE WHEN r.state_id = 19 THEN 1 ELSE 0 END) AS upload_failed,
    SUM(CASE WHEN r.state_id = 21 THEN 1 ELSE 0 END) AS attachment_failed
FROM record r
INNER JOIN client c ON r.client_id = c.client_id
WHERE r.is_deleted = false
  AND r.state_id IN (11,13,14,15,16,17,19,21)
GROUP BY c.name, DATE_TRUNC('month', r.created_at)
ORDER BY c.name, month;
"""

df_unbilled = query(SQL_UNBILLED)
df_unbilled['month'] = pd.to_datetime(df_unbilled['month'], utc=True).dt.normalize().dt.tz_localize(None) + pd.offsets.MonthBegin(0)

# Monthly unbilled totals per client — heatmap
pivot_unb = df_unbilled.pivot_table(index='client_name', columns='month', values='unbilled_records', fill_value=0)
pivot_unb.columns = pivot_unb.columns.strftime('%Y-%m')

fig, axes = plt.subplots(2, 1, figsize=(16, max(8, len(pivot_unb) * 0.8)))

sns.heatmap(pivot_unb.astype(float), annot=True, fmt='.0f', cmap='Purples',
            ax=axes[0], cbar_kws={'label': 'Unbilled records'})
axes[0].set_title('Unbilled Records per Client per Month (state ≥ 11, not in billing states)')
axes[0].set_ylabel('')

# Current month breakdown by state
unbilled_month = df_unbilled[df_unbilled['month'] == df_unbilled['month'].max()].copy()
if not unbilled_month.empty:
    state_cols = ['harvested', 'informed', 'requested', 'downloaded',
                  'accounting_applied', 'accounting_failed', 'upload_failed', 'attachment_failed']
    unbilled_month.set_index('client_name')[state_cols].plot(kind='barh', stacked=True, ax=axes[1], colormap='tab10')
    axes[1].set_title(f'Unbilled Records by State — {REPORT_YEAR}-{REPORT_MONTH:02d}')
    axes[1].set_xlabel('Records'); axes[1].set_ylabel('')
    axes[1].legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout(); plt.show()

total_unbilled = df_unbilled[df_unbilled['month'] == df_unbilled['month'].max()]['unbilled_records'].sum()
print(f'Total unbilled records this month: {total_unbilled:,}')
df_unbilled

## 17. Unbilled Records (state ≥ 11, not billed)

In [ ]:
STATE_NAMES = {
    1: 'INTERCEPTED', 2: 'INTERCEPTION_ERROR', 3: 'INTERNAL_ERROR',
    4: 'READ', 5: 'NOT_READ', 6: 'UNRECOGNIZED', 7: 'TRANSFORMED',
    8: 'PROMPTED', 9: 'NOT_PROMPTED', 10: 'NOT_HARVESTED',
    11: 'HARVESTED', 12: 'DUPLICATE_DETECTED', 13: 'INFORMED',
    14: 'REQUESTED', 15: 'DOWNLOADED', 16: 'ACCOUNTING_LOGIC_APPLIED',
    17: 'ACCOUNTING_LOGIC_FAILED', 18: 'DATA_UPLOADED',
    19: 'DATA_UPLOAD_FAILED', 20: 'ATTACHMENT_UPLOADED', 21: 'ATTACHMENT_UPLOAD_FAILED',
}

SQL_STATE_DIST = """
SELECT
    c.name     AS client_name,
    r.state_id,
    COUNT(*)   AS records
FROM record r
INNER JOIN client c ON r.client_id = c.client_id
WHERE r.is_deleted = false
  AND EXTRACT(YEAR  FROM r.created_at) = %(year)s
  AND EXTRACT(MONTH FROM r.created_at) = %(month)s
GROUP BY c.name, r.state_id
ORDER BY c.name, r.state_id;
"""

df_states = query(SQL_STATE_DIST, {'year': REPORT_YEAR, 'month': REPORT_MONTH})
df_states['state_name'] = df_states['state_id'].map(STATE_NAMES)

pivot_states = df_states.pivot_table(index='client_name', columns='state_name', values='records', fill_value=0)

fig, ax = plt.subplots(figsize=(16, max(5, len(pivot_states) * 0.6)))
pivot_states.plot(kind='barh', stacked=True, ax=ax, colormap='tab20')
ax.set_title(f'State Distribution per Client — {REPORT_YEAR}-{REPORT_MONTH:02d}')
ax.set_xlabel('Records'); ax.set_ylabel('')
ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout(); plt.show()

## 16. State Distribution

In [ ]:
SQL_DUPLICATES = """
SELECT
    c.name AS client_name,
    DATE_TRUNC('month', r.created_at) AS month,
    SUM(CASE WHEN r.state_id = 12  THEN 1 ELSE 0 END) AS duplicates,
    SUM(CASE WHEN r.state_id >= 11 THEN 1 ELSE 0 END) AS total_harvested,
    ROUND(SUM(CASE WHEN r.state_id = 12 THEN 1 ELSE 0 END) * 100.0
        / NULLIF(SUM(CASE WHEN r.state_id >= 11 THEN 1 ELSE 0 END), 0), 1) AS duplicate_pct
FROM record r
INNER JOIN client c ON r.client_id = c.client_id
WHERE r.is_deleted = false
GROUP BY c.name, DATE_TRUNC('month', r.created_at)
ORDER BY c.name, month;
"""

df_dup = query(SQL_DUPLICATES)
df_dup['month']         = pd.to_datetime(df_dup['month'], utc=True).dt.normalize().dt.tz_localize(None) + pd.offsets.MonthBegin(0)
df_dup['duplicate_pct'] = pd.to_numeric(df_dup['duplicate_pct'], errors='coerce')

pivot_dup = df_dup.pivot_table(index='client_name', columns='month', values='duplicate_pct', fill_value=0)
pivot_dup.columns = pivot_dup.columns.strftime('%Y-%m')

fig, ax = plt.subplots(figsize=(14, max(4, len(pivot_dup) * 0.6)))
sns.heatmap(pivot_dup.astype(float), annot=True, fmt='.1f', cmap='Oranges',
            ax=ax, cbar_kws={'label': 'Duplicate %'})
ax.set_title('Duplicate Detection Rate % per Client per Month (of harvested records)')
ax.set_ylabel('')
plt.tight_layout(); plt.show()

## 15. Duplicate Detection Rate

In [ ]:
SQL_SUCCESS_TREND = """
SELECT
    c.name AS client_name,
    DATE_TRUNC('month', r.created_at) AS month,
    COUNT(*) AS total,
    SUM(CASE WHEN r.state_id IN (12,18,20)           THEN 1 ELSE 0 END) AS billed,
    SUM(CASE WHEN r.state_id IN (2,3,5,9,10,17,19,21) THEN 1 ELSE 0 END) AS errors,
    ROUND(SUM(CASE WHEN r.state_id IN (12,18,20)            THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*),0), 1) AS billed_pct,
    ROUND(SUM(CASE WHEN r.state_id IN (2,3,5,9,10,17,19,21) THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*),0), 1) AS error_pct
FROM record r
INNER JOIN client c ON r.client_id = c.client_id
WHERE r.is_deleted = false
GROUP BY c.name, DATE_TRUNC('month', r.created_at)
ORDER BY c.name, month;
"""

df_suc = query(SQL_SUCCESS_TREND)
df_suc['month']      = pd.to_datetime(df_suc['month'], utc=True).dt.normalize().dt.tz_localize(None) + pd.offsets.MonthBegin(0)
df_suc['billed_pct'] = pd.to_numeric(df_suc['billed_pct'], errors='coerce')
df_suc['error_pct']  = pd.to_numeric(df_suc['error_pct'],  errors='coerce')

pivot_billed = df_suc.pivot(index='client_name', columns='month', values='billed_pct')
pivot_error  = df_suc.pivot(index='client_name', columns='month', values='error_pct')
pivot_billed.columns = pivot_billed.columns.strftime('%Y-%m')
pivot_error.columns  = pivot_error.columns.strftime('%Y-%m')

fig, axes = plt.subplots(1, 2, figsize=(18, max(4, len(pivot_billed) * 0.6)))
sns.heatmap(pivot_billed.astype(float), annot=True, fmt='.0f', cmap='Greens',
            ax=axes[0], cbar_kws={'label': 'Billed %'}, vmin=0, vmax=100)
axes[0].set_title('Billed Records % per Client per Month'); axes[0].set_ylabel('')
sns.heatmap(pivot_error.astype(float), annot=True, fmt='.0f', cmap='Reds',
            ax=axes[1], cbar_kws={'label': 'Error %'}, vmin=0, vmax=100)
axes[1].set_title('Error Rate % per Client per Month'); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()

## 14. Success vs Error Rate Trend

In [ ]:
SQL_ERRORS = """
SELECT
    c.name AS client_name,
    COUNT(*) AS total_records,
    SUM(CASE WHEN r.state_id = 2  THEN 1 ELSE 0 END) AS interception_error,
    SUM(CASE WHEN r.state_id = 3  THEN 1 ELSE 0 END) AS internal_error,
    SUM(CASE WHEN r.state_id = 5  THEN 1 ELSE 0 END) AS not_read,
    SUM(CASE WHEN r.state_id = 9  THEN 1 ELSE 0 END) AS not_prompted,
    SUM(CASE WHEN r.state_id = 10 THEN 1 ELSE 0 END) AS not_harvested,
    SUM(CASE WHEN r.state_id = 17 THEN 1 ELSE 0 END) AS accounting_failed,
    SUM(CASE WHEN r.state_id = 19 THEN 1 ELSE 0 END) AS upload_failed,
    SUM(CASE WHEN r.state_id = 21 THEN 1 ELSE 0 END) AS attachment_failed,
    SUM(CASE WHEN r.state_id IN (2,3,5,9,10,17,19,21) THEN 1 ELSE 0 END) AS total_errors,
    ROUND(SUM(CASE WHEN r.state_id IN (2,3,5,9,10,17,19,21) THEN 1 ELSE 0 END)
        * 100.0 / NULLIF(COUNT(*), 0), 1) AS error_pct
FROM record r
INNER JOIN client c ON r.client_id = c.client_id
WHERE r.is_deleted = false
  AND EXTRACT(YEAR  FROM r.created_at) = %(year)s
  AND EXTRACT(MONTH FROM r.created_at) = %(month)s
GROUP BY c.name
ORDER BY error_pct DESC;
"""

df_errors = query(SQL_ERRORS, {'year': REPORT_YEAR, 'month': REPORT_MONTH})
df_errors['error_pct'] = pd.to_numeric(df_errors['error_pct'], errors='coerce')

error_cols = ['interception_error', 'internal_error', 'not_read', 'not_prompted',
              'not_harvested', 'accounting_failed', 'upload_failed', 'attachment_failed']

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(df_errors) * 0.5)))
df_errors.set_index('client_name')[error_cols].plot(kind='barh', stacked=True, ax=axes[0])
axes[0].set_title(f'Error Breakdown per Client — {REPORT_YEAR}-{REPORT_MONTH:02d}')
axes[0].set_xlabel('Records'); axes[0].set_ylabel('')
axes[0].legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
axes[1].barh(df_errors['client_name'], df_errors['error_pct'], color='tomato')
axes[1].set_title('Error Rate % per Client'); axes[1].set_xlabel('Error %')
axes[1].bar_label(axes[1].containers[0], fmt='%.1f%%', padding=3)
plt.tight_layout(); plt.show()
df_errors

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime

PDF_PATH = f'client_statistics_{REPORT_YEAR}_{REPORT_MONTH:02d}.pdf'

def _fig(*args, **kwargs):
    return plt.subplots(*args, **{**{'figsize': (14, 6)}, **kwargs})

def _fmt(val):
    """Format a cell value: numbers get thousands separator, strings stay as-is."""
    try:
        n = float(val)
        if n == int(n):
            return f'{int(n):,}'
        return f'{n:,.2f}'
    except (TypeError, ValueError):
        return str(val) if val is not None else ''

with PdfPages(PDF_PATH) as pdf:

    # ── Cover page ─────────────────────────────────────────────
    fig, ax = _fig()
    ax.axis('off')
    ax.text(0.5, 0.65, 'Client Statistics Report',
            ha='center', va='center', fontsize=28, fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.50, f'Period: {REPORT_YEAR}-{REPORT_MONTH:02d}',
            ha='center', va='center', fontsize=18, transform=ax.transAxes)
    ax.text(0.5, 0.38, f'Generated: {datetime.today().strftime("%Y-%m-%d")}',
            ha='center', va='center', fontsize=13, color='grey', transform=ax.transAxes)
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 1. Current month summary table ─────────────────────────
    col_align = ['left', 'right', 'right', 'right']   # per column

    # Format all values
    fmt_rows = [[_fmt(v) for v in row] for row in df_summary.values]

    fig, ax = _fig(figsize=(14, max(3, len(df_summary) * 0.45)))
    ax.axis('off')
    tbl = ax.table(
        cellText=fmt_rows,
        colLabels=df_summary.columns,
        loc='center'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.5)

    n_rows = len(fmt_rows)
    n_cols = len(df_summary.columns)

    for (row, col), cell in tbl.get_celld().items():
        # Header row
        if row == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
            cell.set_edgecolor('white')
            cell.get_text().set_ha(col_align[col] if col < len(col_align) else 'right')
        # ZZ TOTAL row (last data row)
        elif row == n_rows:
            cell.set_facecolor('#ecf0f1')
            cell.set_text_props(fontweight='bold')
            cell.set_edgecolor('#bdc3c7')
            cell.get_text().set_ha(col_align[col] if col < len(col_align) else 'right')
        # Regular rows — alternating background
        else:
            cell.set_facecolor('#ffffff' if row % 2 == 0 else '#f8f9fa')
            cell.set_edgecolor('#dee2e6')
            cell.get_text().set_ha(col_align[col] if col < len(col_align) else 'right')

        # Add right padding for numeric cells
        if col > 0:
            cell.PAD = 0.08

    ax.set_title(f'Current Month Summary — {REPORT_YEAR}-{REPORT_MONTH:02d}',
                 fontsize=14, fontweight='bold', pad=16)
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 2. Records & revenue bar charts ────────────────────────
    fig, axes = _fig(1, 2, figsize=(14, 5))
    axes[0].barh(clients.client_name, clients.records_created)
    axes[0].set_title(f'Records Created — {REPORT_YEAR}-{REPORT_MONTH:02d}'); axes[0].set_xlabel('Records')
    axes[1].barh(clients.client_name, clients.monthly_price_czk)
    axes[1].set_title(f'Monthly Price (CZK) — {REPORT_YEAR}-{REPORT_MONTH:02d}'); axes[1].set_xlabel('CZK')
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 3. MoM growth heatmap ──────────────────────────────────
    pivot = df_mom.pivot(index='client_name', columns='month', values='revenue_growth_pct')
    pivot.columns = pivot.columns.strftime('%Y-%m')
    fig, ax = _fig(figsize=(14, max(4, len(pivot) * 0.6)))
    sns.heatmap(pivot.astype(float), annot=True, fmt='.0f', center=0, cmap='RdYlGn',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'Revenue growth %'})
    ax.set_title('Month-over-Month Revenue Growth % per Client'); ax.set_ylabel(''); ax.set_xlabel('')
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 4. Revenue trend lines ─────────────────────────────────
    fig, axes = _fig(1, 2, figsize=(16, 5))
    for client, grp in df_trend.groupby('client_name'):
        axes[0].plot(grp['month'], grp['records_created'], marker='o', label=client)
        axes[1].plot(grp['month'], grp['monthly_price_czk'], marker='o', label=client)
    axes[0].set_title('Records Created per Month per Client'); axes[0].set_ylabel('Records'); axes[0].legend(fontsize=7)
    axes[1].set_title('Revenue (CZK) per Month per Client'); axes[1].set_ylabel('CZK')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}')); axes[1].legend(fontsize=7)
    for ax in axes: ax.tick_params(axis='x', rotation=30)
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 5. Cumulative YTD revenue ──────────────────────────────
    fig, ax = _fig(figsize=(12, 5))
    ax.bar(df_cum['month'], df_cum['monthly_revenue_czk'], width=20, alpha=0.6, label='Monthly revenue')
    ax2 = ax.twinx()
    ax2.plot(df_cum['month'], df_cum['cumulative_revenue_czk'], color='tab:red', marker='o', label='Cumulative YTD')
    ax.set_title(f'Monthly & Cumulative YTD Revenue {REPORT_YEAR} (CZK)')
    ax.set_ylabel('Monthly (CZK)'); ax2.set_ylabel('Cumulative (CZK)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.tick_params(axis='x', rotation=30)
    l1, lb1 = ax.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, lb1+lb2, loc='upper left')
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 6. Daily heatmap ───────────────────────────────────────
    pivot_d = df_daily.pivot_table(index='client_name', columns='day', values='records_created', fill_value=0)
    pivot_d.columns = pivot_d.columns.strftime('%d')
    fig, ax = _fig(figsize=(16, max(4, len(pivot_d) * 0.6)))
    sns.heatmap(pivot_d, cmap='YlOrRd', linewidths=0.3, ax=ax, cbar_kws={'label': 'Records'})
    ax.set_title(f'Daily Document Volume per Client — {REPORT_YEAR}-{REPORT_MONTH:02d}')
    ax.set_xlabel('Day'); ax.set_ylabel(''); ax.tick_params(axis='x', rotation=0)
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 7. Harvest efficiency ──────────────────────────────────
    fig, ax = _fig(figsize=(10, max(4, len(df_eff) * 0.5)))
    bars = ax.barh(df_eff['client_name'], df_eff['records_per_job'])
    ax.bar_label(bars, fmt='%.1f', padding=3)
    ax.set_title('Records per Harvest Job (all time)'); ax.set_xlabel('Records / Job')
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 8. Pareto ──────────────────────────────────────────────
    fig, ax = _fig(figsize=(12, 5))
    ax.bar(df_pareto['client_name'], df_pareto['records_created'], alpha=0.7)
    ax2 = ax.twinx()
    ax2.plot(df_pareto['client_name'], df_pareto['cumulative_pct'], color='tab:red', marker='o')
    ax2.axhline(80, color='tab:red', linestyle='--', alpha=0.4, label='80 %')
    ax.set_title('Pareto: Records per Client (all time)'); ax.set_ylabel('Records')
    ax2.set_ylabel('Cumulative %'); ax2.set_ylim(0, 105)
    ax.tick_params(axis='x', rotation=30); ax2.legend()
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 9. LLM cost heatmap ────────────────────────────────────
    pivot_llm = df_llm.pivot_table(index='client_name', columns='month', values='total_cost_eur', fill_value=0)
    pivot_llm.columns = pivot_llm.columns.strftime('%Y-%m')
    fig, ax = _fig(figsize=(14, max(4, len(pivot_llm) * 0.6)))
    sns.heatmap(pivot_llm, annot=True, fmt='.2f', cmap='Blues', ax=ax, cbar_kws={'label': 'Cost (EUR)'})
    ax.set_title('LLM Cost (EUR) per Client per Month'); ax.set_ylabel('')
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 10. Model usage ────────────────────────────────────────
    fig, axes = _fig(1, 2, figsize=(16, 5))
    pivot_calls = df_models.pivot_table(index='month', columns='model_name', values='llm_calls', fill_value=0)
    pivot_calls.index = pivot_calls.index.strftime('%Y-%m')   # categorical labels -> avoids ts-plot freq error
    pivot_calls.plot(kind='bar', stacked=True, ax=axes[0])
    axes[0].set_title('LLM Calls per Model per Month'); axes[0].set_ylabel('Calls')
    axes[0].tick_params(axis='x', rotation=30); axes[0].legend(fontsize=8)
    pivot_cost = df_models.pivot_table(index='month', columns='model_name', values='total_cost_eur', fill_value=0)
    pivot_cost.index = pivot_cost.index.strftime('%Y-%m')
    pivot_cost.plot(kind='bar', stacked=True, ax=axes[1])
    axes[1].set_title('LLM Cost (EUR) per Model per Month'); axes[1].set_ylabel('EUR')
    axes[1].tick_params(axis='x', rotation=30); axes[1].legend(fontsize=8)
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 11. LLM cost trend ─────────────────────────────────────
    fig, ax = _fig(figsize=(12, 5))
    ax.bar(df_llm_trend['month'], df_llm_trend['cost_eur'], width=20, alpha=0.6, label='Monthly cost')
    ax2 = ax.twinx()
    ax2.plot(df_llm_trend['month'], df_llm_trend['cumulative_cost_eur'], color='tab:red', marker='o', label='Cumulative')
    ax.set_title('Monthly & Cumulative LLM Cost (EUR)')
    ax.set_ylabel('Monthly cost (EUR)'); ax2.set_ylabel('Cumulative (EUR)')
    ax.tick_params(axis='x', rotation=30)
    l1, lb1 = ax.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1+l2, lb1+lb2, loc='upper left')
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 12. Pipeline funnel ────────────────────────────────────
    fig, ax = _fig(figsize=(14, max(5, len(df_funnel) * 0.5)))
    x = range(len(stages))
    for _, row in df_funnel.iterrows():
        ax.plot(x, [row[s] for s in stages], marker='o', label=row['client_name'])
    ax.set_xticks(range(len(stages))); ax.set_xticklabels(stage_labels)
    ax.set_title(f'Pipeline Funnel per Client — {REPORT_YEAR}-{REPORT_MONTH:02d}'); ax.set_ylabel('Records')
    ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 13. Error analysis ─────────────────────────────────────
    fig, axes = _fig(1, 2, figsize=(16, max(5, len(df_errors) * 0.5)))
    df_errors.set_index('client_name')[error_cols].plot(kind='barh', stacked=True, ax=axes[0])
    axes[0].set_title(f'Error Breakdown per Client — {REPORT_YEAR}-{REPORT_MONTH:02d}')
    axes[0].set_xlabel('Records'); axes[0].set_ylabel('')
    axes[0].legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    axes[1].barh(df_errors['client_name'], df_errors['error_pct'], color='tomato')
    axes[1].set_title('Error Rate % per Client'); axes[1].set_xlabel('Error %')
    axes[1].bar_label(axes[1].containers[0], fmt='%.1f%%', padding=3)
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 14. Success vs error trend ─────────────────────────────
    fig, axes = _fig(1, 2, figsize=(18, max(4, len(pivot_billed) * 0.6)))
    sns.heatmap(pivot_billed.astype(float), annot=True, fmt='.0f', cmap='Greens',
                ax=axes[0], cbar_kws={'label': 'Billed %'}, vmin=0, vmax=100)
    axes[0].set_title('Billed Records % per Client per Month'); axes[0].set_ylabel('')
    sns.heatmap(pivot_error.astype(float), annot=True, fmt='.0f', cmap='Reds',
                ax=axes[1], cbar_kws={'label': 'Error %'}, vmin=0, vmax=100)
    axes[1].set_title('Error Rate % per Client per Month'); axes[1].set_ylabel('')
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 15. Duplicate detection rate ───────────────────────────
    fig, ax = _fig(figsize=(14, max(4, len(pivot_dup) * 0.6)))
    sns.heatmap(pivot_dup.astype(float), annot=True, fmt='.1f', cmap='Oranges',
                ax=ax, cbar_kws={'label': 'Duplicate %'})
    ax.set_title('Duplicate Detection Rate % per Client per Month (of harvested records)')
    ax.set_ylabel('')
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 16. State distribution ─────────────────────────────────
    fig, ax = _fig(figsize=(16, max(5, len(pivot_states) * 0.6)))
    pivot_states.plot(kind='barh', stacked=True, ax=ax, colormap='tab20')
    ax.set_title(f'State Distribution per Client — {REPORT_YEAR}-{REPORT_MONTH:02d}')
    ax.set_xlabel('Records'); ax.set_ylabel('')
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    # ── 17. Unbilled records ───────────────────────────────────
    fig, axes = plt.subplots(2, 1, figsize=(16, max(8, len(pivot_unb) * 0.8)))
    sns.heatmap(pivot_unb.astype(float), annot=True, fmt='.0f', cmap='Purples',
                ax=axes[0], cbar_kws={'label': 'Unbilled records'})
    axes[0].set_title('Unbilled Records per Client per Month (state ≥ 11, not billed)'); axes[0].set_ylabel('')
    if not unbilled_month.empty:
        unbilled_month.set_index('client_name')[state_cols].plot(kind='barh', stacked=True, ax=axes[1], colormap='tab10')
        axes[1].set_title(f'Unbilled Records by State — {REPORT_YEAR}-{REPORT_MONTH:02d}')
        axes[1].set_xlabel('Records'); axes[1].set_ylabel('')
        axes[1].legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)

    pdf.infodict()['Title']   = f'Client Statistics {REPORT_YEAR}-{REPORT_MONTH:02d}'
    pdf.infodict()['Author']  = 'Granton AI'
    pdf.infodict()['Subject'] = 'Board Report'

print(f'Saved → {PDF_PATH} (17 sections)')

## 12. Export to PDF